# Module 02: Blockchain Indexing with Envio

In this lesson, we will learn how to extract, decode, and store raw blockchain events directly into our PostgreSQL database using a modern indexing tool called **Envio**.

---

## 1. What is Envio and What Problem Does It Solve?

Smart contracts emit **events** (logs) when actions happen (e.g., when a user transfers tokens). On the blockchain, these events are stored as raw, encoded hexadecimal strings. 

**Envio** is a developer-friendly framework that tracks the blockchain, catches the specified events, decodes them instantly using the contract's **ABI**, and saves them directly to structured tables in your PostgreSQL database.

## 2. Installing Prerequisites & Envio

Before running Envio, you need a few standard developer tools installed on your computer. Here is the easiest way to set them up:

### Step 1: Install Node.js (Includes npm)
Node.js runs the JavaScript engine that Envio relies on. 
* **Mac/Windows**: Download the installer from the official website: [Node.js Downloads](https://nodejs.org/)
* **Linux (Ubuntu/WSL)**: Run the following commands in your terminal:
  ```bash
  sudo apt update
  sudo apt install -y nodejs npm
  ```

### Step 2: Install pnpm
Envio projects are optimized to use **pnpm** (a faster alternative to npm). Once Node.js is installed, run this single command to install pnpm globally:
```bash
npm install -g pnpm
```

### Step 3: Install Envio
Now install the Envio CLI framework globally on your computer:
```bash
npm install -g envio
```

To verify everything installed correctly, check the versions in your terminal:
```bash
node --version
pnpm --version
envio --version
```

## 3. Initializing an Envio Project

Before editing configuration files, we need to initialize a fresh Envio project workspace. Create a new directory for your indexer, navigate into it, and run the initialization command:

```bash
mkdir my-token-indexer
cd my-token-indexer
envio init
```

Once you run `envio init`, the CLI will launch an interactive wizard. Choose these options:

1. **Choose a name for your indexer**: Enter `token-indexer`.

2. **Blockchain network**: Select `Evm`

3. **Choose an initialization template**: Select `ERC20`

4. **Envio API Token**: Go to the official [Envio Website](https://envio.dev/) and log in to generate an API token under your account settings. Paste the API token here and press Enter.

## 4. Configuring a Token Transfer Indexer

An Envio indexer relies on three main files inside the generated project folder:

### A. The Configuration (`config.yaml`) | Find code [here](https://github.com/Analytic-Sages/data-engineering-course/blob/main/Module-02/files/config.yaml)
Defines the networks and contract addresses to listen to.

```bash
name: erc20-indexer
description: ERC-20 Indexer
contracts:
  - name: ERC20
    handler: src/handlers/ERC20.ts
    events:
      - event: "Transfer(address indexed from, address indexed to, uint256 value)"
        field_selection:
          transaction_fields:
            - hash
chains:
  - id: 1 # Ethereum Mainnet
    start_block: 18000000
    contracts:
      - name: ERC20
        address: 
          - "0xA0b86991c6218b36c1d19D4a2e9Eb0cE3606eB48" # USDC
          - "0xdac17f958d2ee523a2206206994597c13d831ec7" # USDT
          - "0xc02aaa39b223fe8d0a0e5c4f27ead9083c756cc2" # WETH

```
### B. The Database Schema (`schema.graphql`) | find code [here](https://github.com/Analytic-Sages/data-engineering-course/blob/main/Module-02/files/schema.graphql)
Defines the table structure in your PostgreSQL database.

```bash
type Transfer {
  id: ID!          # Unique identifier for each transfer (usually transaction hash + log index)
  from: String!    # Sender address
  to: String!      # Recipient address
  value: BigInt!   # Transfer amount
  blockNumber: Int! 
  txhash: String!
  blockTimestamp: Int!
  tokenAddress: String!
}
```
### C. The Event Handler (`src/handlers/ERC20.ts`) | Find code [here](https://github.com/Analytic-Sages/data-engineering-course/blob/main/Module-02/files/ERC20.ts)
Transforms raw events into database entities.

```typescript
import { indexer, type Transfer } from "envio";

indexer.onEvent(
  { contract: "ERC20", event: "Transfer" },
  async ({ event, context }) => {
    const transferObject: Transfer = {
      id: `${event.transaction.hash}-${event.logIndex}`,
      from: event.params.from,
      to: event.params.to,
      value: event.params.value,
      blockNumber: event.block.number,
      txhash: event.transaction.hash,
      blockTimestamp: event.block.timestamp,
      tokenAddress: event.srcAddress,
    };

    // Save the transfer to our database
    context.Transfer.set(transferObject);
  },
);



```

## 5. Running Modes: Natively vs. Docker

We have two ways of running our Envio indexer: **Natively (without Docker configuration)** and **Containerized (with custom PostgreSQL warehouse)**.

### Option A: Running Natively (Without Docker configuration)
If you just want to develop and test your indexer quickly without setting up a custom database, you can run Envio directly on your host machine. 

In this mode:
* Envio will automatically spin up its own temporary Postgres database in Docker in the background.
* **You do not need to specify any Postgres credentials inside `.env`**.
* Your `.env` file only requires the API token:
  ```env
  ENVIO_API_TOKEN="your-api-token"
  ```

To start indexing natively, run:
```bash
envio codegen
envio dev
```

---

### Option B: Running with Docker (To a Custom PostgreSQL Warehouse)
When moving to production or running your own analytical warehouse, you want Envio to save data directly into your custom database.

In this mode:
* We run all services (Postgres, Hasura, and the Envio indexer) together in a shared Docker Compose network.
* **You must define your database credentials inside `.env`** so the services can link up:

Create the `.env` file in the indexer directory
  ```bash
  ENVIO_API_TOKEN="your-api-token"
  ENVIO_PG_HOST=db
  ENVIO_PG_PORT=5432
  ENVIO_PG_USER=root
  ENVIO_PG_PASSWORD=root
  ENVIO_PG_DATABASE=data-set
  ENVIO_PG_PUBLIC_SCHEMA=analytics
  COMPOSE_LOG_FORMAT=raw
  ```

Create a dockerfile and docker compose file in the indexer directory. 

copy the code here [Dockerfile](https://github.com/Analytic-Sages/data-engineering-course/blob/main/Module-02/files/Dockerfile) | [Docker-compose file](https://github.com/Analytic-Sages/data-engineering-course/blob/main/Module-02/files/docker-compose.yml)

To start Envio with your custom database warehouse in Docker, run:
```bash
docker compose run --build indexer
```

Using `docker compose run --build indexer` builds the updated container files (which automatically handles code generation via the `Dockerfile`) and starts Envio interactively with full TUI support inside your active terminal console.